In [ ]:
# Permite importar src/ desde la carpeta notebooks/
import sys
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import DIR_DATA_RAW

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid")

In [ ]:
from src.config import DIR_DATA_RAW, DIR_DATA_PROC
from src.telemetria import asignar_viaje, cruzar_nombres

viajes = pd.read_parquet(DIR_DATA_PROC / "viajes.parquet")
tel = pd.concat([pd.read_parquet(a) for a in (DIR_DATA_RAW / "telemetria").glob("*.parquet")], ignore_index=True)
tel["fecha"] = pd.to_datetime(tel["fecha"])
tel["ep"] = tel["ep"].astype(str).str.strip().str.upper()

# Limpieza mínima ya acordada
antes = len(tel)
tel = tel.drop_duplicates(["ep", "fecha"])
tel = tel[(tel["Latitud"].abs() < 89) & (tel["Longitud"].abs() < 179)]
tel = tel[tel["rpm"].between(300, 1400) & (tel["gal_h"] < 6553)]
print(f"Telemetría: {antes:,} → {len(tel):,} registros")

# 1. ¿Las coordenadas quedaron bien? Deben verse en la costa de Perú
print(tel.groupby("origen")[["Latitud", "Longitud"]].median().round(2))

# 2. Nombres
n = cruzar_nombres(tel, viajes)
display(n[n["en_telemetria"] & ~n["en_faenas"]])

# 3. Asignación de viaje
tel_v = asignar_viaje(tel, viajes)
print(f"Registros con viaje: {tel_v['IdViaje'].notna().mean():.1%}")

# 4. % con viaje por mes y embarcación
tel_v["mes"] = tel_v["fecha"].dt.to_period("M")
pct = tel_v.pivot_table(index="ep", columns="mes", values="IdViaje",
                        aggfunc=lambda s: s.notna().mean() * 100)
plt.figure(figsize=(16, 6))
sns.heatmap(pct, cmap="Blues", vmin=0, vmax=100, annot=True, fmt=".0f", cbar_kws={"label": "% con viaje"})
plt.title("% de registros de telemetría dentro de un viaje")
plt.tight_layout()

guardar = tel_v.drop(columns=["mes"]).copy()
guardar["id"] = guardar["id"].astype(str)          # una sola forma: texto
guardar.to_parquet(DIR_DATA_PROC / "telemetria_viajes.parquet", index=False)
print("Guardado:", len(guardar), "registros")

In [ ]:
crudo = pd.concat([pd.read_parquet(a) for a in (DIR_DATA_RAW / "telemetria").glob("*.parquet")], ignore_index=True)

# 1. ¿Cuándo se extrajo cada archivo y cómo vienen las coordenadas?
import os, datetime
for a in sorted((DIR_DATA_RAW / "telemetria").glob("*.parquet")):
    print(a.name, datetime.datetime.fromtimestamp(os.path.getmtime(a)))
print(crudo.groupby("origen")[["Latitud", "Longitud", "rpm", "gal_h"]].median())

# 2. ¿Qué filtro elimina cada fuente?
c = crudo.copy()
c["coord_ok"] = (c["Latitud"].abs() < 89) & (c["Longitud"].abs() < 179)
c["rpm_ok"] = pd.to_numeric(c["rpm"], errors="coerce").between(300, 1400)
c["galh_ok"] = pd.to_numeric(c["gal_h"], errors="coerce") < 6553
display(c.groupby("origen")[["coord_ok", "rpm_ok", "galh_ok"]].mean().round(3))

In [ ]:
tel_v = pd.read_parquet(DIR_DATA_PROC / "telemetria_viajes.parquet")
tel_v["mes"] = tel_v["fecha"].dt.to_period("M").astype(str)
viajes["mes"] = viajes["ZarpeCorregido"].dt.to_period("M").astype(str)

cob = (tel_v.groupby(["ep", "mes"])
       .agg(registros=("fecha", "size"),
            pct_con_viaje=("IdViaje", lambda s: round(100 * s.notna().mean(), 1)),
            viajes_con_telemetria=("IdViaje", "nunique"))
       .reset_index())
n_viajes = viajes.groupby(["Embarcacion", "mes"]).size().rename("viajes_en_faenas")
cob = cob.merge(n_viajes, left_on=["ep", "mes"], right_index=True, how="left").fillna({"viajes_en_faenas": 0})

# Regla: un mes de una embarcación es confiable si tiene viajes y al menos 40 % de sus registros cae en ellos
def clasificar(r):
    if r.viajes_en_faenas == 0 and r.pct_con_viaje == 0:
        return "sin faenas (veda o faltantes)"
    if r.pct_con_viaje >= 95 and r.registros > 200:
        return "revisar: casi todo en viaje"
    if r.pct_con_viaje >= 40:
        return "confiable"
    return "revisar: pocas faenas"
cob["estado"] = cob.apply(clasificar, axis=1)

display(cob["estado"].value_counts())
display(cob[cob["estado"].str.startswith("revisar")].sort_values(["ep", "mes"]))

cob.to_parquet(DIR_DATA_PROC / "cobertura.parquet", index=False)
cob[cob["estado"] != "confiable"].to_csv(DIR_DATA_PROC / "faenas_faltantes_para_reportar.csv", index=False)

# Registros que usaremos para entrenar
ok = cob.loc[cob["estado"] == "confiable", ["ep", "mes"]]
tel_ok = tel_v.merge(ok, on=["ep", "mes"])
print(f"Registros en meses confiables: {len(tel_ok):,} de {len(tel_v):,} "
      f"({len(tel_ok)/len(tel_v):.0%}); con viaje: {tel_ok['IdViaje'].notna().mean():.0%}")

In [ ]:
from src.puertos import estimar_puertos, distancia_puerto, estado_fuera_de_viaje

puertos = estimar_puertos(tel_v, viajes)
display(puertos)

tel_v = distancia_puerto(tel_v, puertos)
tel_v["Estado"] = estado_fuera_de_viaje(tel_v)

# Horas por estado (cada registro ≈ 15 min)
horas = (tel_v.groupby(["ep", "mes", "Estado"]).size() / 4).unstack(fill_value=0).round(0)
display(horas.loc[["BLANDI", "SANTA ADELA II", "POLAR I", "MARY"]])

# ¿Dónde está la embarcación cuando navega sin faena? Una muestra de un caso dudoso
m = tel_v[(tel_v["ep"] == "BLANDI") & (tel_v["mes"] == "2025-07")]
fig, ax = plt.subplots(figsize=(7, 8))
for est, color in [("puerto", "gray"), ("en viaje", "#0065AC"), ("mar sin faena", "#F08300")]:
    g = m[m["Estado"] == est]
    ax.scatter(g["Longitud"], g["Latitud"], s=4, color=color, label=f"{est} ({len(g)})")
ax.scatter(puertos["Longitud"], puertos["Latitud"], marker="s", color="black", s=30, label="puertos")
ax.set_aspect("equal"); ax.legend(); ax.set_title("BLANDI, jul-2025")

tel_v.drop(columns=["mes"]).assign(id=lambda d: d["id"].astype(str)).to_parquet(
    DIR_DATA_PROC / "telemetria_viajes.parquet", index=False)